# C-1N v0.11 STAND shove diagnostics

This notebook reads compact **Telemetry v1** records made during the visual eight-direction shove suite. It does not run or modify C-1N.

## Record the visual experiment

From the repository root:

`python interact.py --seconds 10 --shove-suite telemetry/shoves`

The MuJoCo viewer shows and records `0`, `0.25`, `0.5`, `0.75`, and `1.0 mg` cases at each of eight world-frame directions: `0°` through `315°` in 45° steps. It plots the applied shove and response live at 50 Hz. Each case starts from reset. The force lasts 200 ms.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

TRACE_DIRECTORY = Path('../telemetry/shoves')
DIRECTION_DEG = 135
LABELS = ('0mg', '0.25mg', '0.5mg', '0.75mg', '1mg')

traces = {}
for label in LABELS:
    path = TRACE_DIRECTORY / f'{DIRECTION_DEG:03d}deg' / f'{label}.npz'
    if path.exists():
        with np.load(path) as archive:
            trace = {name: archive[name] for name in archive.files}
        trace['metadata'] = json.loads(str(trace.pop('metadata_json')))
        traces[label] = trace

print(f'Loaded {len(traces)} of {len(LABELS)} Telemetry v1 records at {DIRECTION_DEG}°.')
if traces:
    metadata = next(iter(traces.values()))['metadata']
    print(f"C-1N {metadata['c1n_iteration']}; sample interval: {metadata['sample_interval_s']:.3f} s")
    print(f"m = {metadata['mass_kg']:.3f} kg; g = {metadata['gravity_m_per_s2']:.3f} m/s²; mg = {metadata['weight_n']:.3f} N")

## Compare the same signals shown live

The viewer makes a single run legible. These overlays answer how the response changes with shove magnitude.

In [ ]:
if traces:
    figure, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
    for label, trace in traces.items():
        time_s = trace['time_s']
        axes[0].plot(time_s, trace['force_along_shove_n'], label=f'{label} force')
        axes[1].plot(time_s, trace['torso_displacement_along_shove_m'], label=label)
        axes[2].plot(time_s, trace['support_margin_m'], label=label)
    axes[0].set(title='Applied force along shove direction', ylabel='force (N)')
    axes[1].set(title='Torso displacement along shove direction', ylabel='displacement (m)')
    axes[2].axhline(0.0, color='black', linewidth=0.8)
    axes[2].set(title='Support margin', xlabel='time (s)', ylabel='margin (m)')
    for axis in axes:
        axis.legend()
    figure.tight_layout()

## Inspect load redistribution

This answers which leg pair changes load first as shove magnitude rises.

In [ ]:
if traces:
    figure, axes = plt.subplots(len(traces), 1, figsize=(10, 3 * len(traces)), sharex=True, squeeze=False)
    for axis, (label, trace) in zip(axes[:, 0], traces.items()):
        axis.plot(trace['time_s'], trace['front_pair_load_n'], label='front')
        axis.plot(trace['time_s'], trace['middle_pair_load_n'], label='middle')
        axis.plot(trace['time_s'], trace['rear_pair_load_n'], label='rear')
        axis.set(title=f'{label}: normal load by leg pair', ylabel='load (N)')
        axis.legend()
    axes[-1, 0].set_xlabel('time (s)')
    figure.tight_layout()

## Next experiment

Before changing the model or controller, write: `At ___ mg, I predict ___ happens before ___ because ___.`